# Data Processing: Yahoo Finance + Coffee C COT Join

This notebook joins Arabica Coffee C Yahoo Finance OHLCV data with filtered Coffee C COT features.

Join behavior:
- Uses a **full outer join** on date.
- Keeps every Yahoo trading date.
- Keeps every COT report date even when it does not exist in Yahoo Finance trading dates.
- Sorts the final output in increasing date order.
- Writes outputs into `data/centralData/`.


In [2]:
import pandas as pd
from pathlib import Path

YAHOO_PRICE_FILE = Path("data/yahoo/arabica_coffee_futures_history.csv")
COT_FEATURE_FILE = Path("data/COT") / "coffee_c_all_cot_data.csv"
CENTRAL_DATA_DIR = Path("data/centralData")

CENTRAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Yahoo input: {YAHOO_PRICE_FILE}")
print(f"COT input: {COT_FEATURE_FILE}")
print(f"Output folder: {CENTRAL_DATA_DIR}")


Yahoo input: data/yahoo/arabica_coffee_futures_history.csv
COT input: data/COT/coffee_c_all_cot_data.csv
Output folder: data/centralData


In [ ]:
def load_yahoo_price_data(path=YAHOO_PRICE_FILE):
    yahoo = pd.read_csv(path)
    yahoo["Date"] = pd.to_datetime(yahoo["Date"], errors="coerce").dt.normalize()
    yahoo = yahoo.dropna(subset=["Date"]).copy()

    numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
    for col in numeric_cols:
        if col in yahoo.columns:
            yahoo[col] = pd.to_numeric(yahoo[col], errors="coerce")

    yahoo = yahoo.sort_values("Date").drop_duplicates(subset=["Date"], keep="last")
    return yahoo


def load_cot_feature_data(path=COT_FEATURE_FILE):
    cot = pd.read_csv(path, low_memory=False)
    cot_report_date = pd.to_datetime(
        cot["Report_Date_as_MM_DD_YYYY"], errors="coerce"
    ).dt.normalize()
    cot = pd.concat([cot, pd.DataFrame({"cot_report_date": cot_report_date})], axis=1)
    cot = cot.dropna(subset=["cot_report_date"]).copy()
    return cot


yahoo_prices = load_yahoo_price_data()
cot_features = load_cot_feature_data()

print(f"Yahoo rows: {len(yahoo_prices):,}")
print(f"Yahoo date range: {yahoo_prices['Date'].min().date()} to {yahoo_prices['Date'].max().date()}")
print(f"COT rows: {len(cot_features):,}")
print(f"COT date range: {cot_features['cot_report_date'].min().date()} to {cot_features['cot_report_date'].max().date()}")


In [ ]:
def add_yahoo_derived_features(yahoo):
    yahoo = yahoo.sort_values("Date").copy()

    yahoo["return_1d"] = yahoo["Close"].pct_change()
    yahoo["return_5d"] = yahoo["Close"].pct_change(5)
    yahoo["range_pct"] = (yahoo["High"] - yahoo["Low"]) / yahoo["Close"]
    yahoo["close_to_open_pct"] = (yahoo["Close"] - yahoo["Open"]) / yahoo["Open"]
    yahoo["volume_change_pct"] = yahoo["Volume"].pct_change()
    yahoo["moving_avg_5"] = yahoo["Close"].rolling(5).mean()
    yahoo["moving_avg_20"] = yahoo["Close"].rolling(20).mean()
    yahoo["close_vs_ma_20"] = yahoo["Close"] / yahoo["moving_avg_20"] - 1

    yahoo["future_close_1d"] = yahoo["Close"].shift(-1)
    yahoo["future_close_5d"] = yahoo["Close"].shift(-5)
    yahoo["future_return_1d"] = yahoo["future_close_1d"] / yahoo["Close"] - 1
    yahoo["future_return_5d"] = yahoo["future_close_5d"] / yahoo["Close"] - 1
    yahoo["future_direction_5d"] = yahoo["future_return_5d"] > 0

    return yahoo


def add_cot_derived_features(cot):
    cot = cot.copy()

    numeric_prefixes = (
        "Open_Interest",
        "Prod_Merc",
        "Swap",
        "M_Money",
        "Other_Rept",
        "Tot_Rept",
        "NonRept",
        "Change_in",
        "Pct_of",
        "Traders",
        "Conc",
        "NonComm",
        "Comm",
    )
    numeric_columns = [
        col for col in cot.columns
        if col.startswith(numeric_prefixes)
    ]
    for col in numeric_columns:
        cot[col] = pd.to_numeric(cot[col], errors="coerce")

    formulas = {
        "managed_money_net": ("M_Money_Positions_Long_ALL", "M_Money_Positions_Short_ALL"),
        "managed_money_net_pct_oi": ("Pct_of_OI_M_Money_Long_All", "Pct_of_OI_M_Money_Short_All"),
        "producer_merchant_net": ("Prod_Merc_Positions_Long_ALL", "Prod_Merc_Positions_Short_ALL"),
        "commercial_net": ("Comm_Positions_Long_All", "Comm_Positions_Short_All"),
        "noncommercial_net": ("NonComm_Positions_Long_All", "NonComm_Positions_Short_All"),
        "nonreportable_net": ("NonRept_Positions_Long_All", "NonRept_Positions_Short_All"),
        "swap_dealer_net": ("Swap_Positions_Long_All", "Swap__Positions_Short_All"),
        "other_reportable_net": ("Other_Rept_Positions_Long_ALL", "Other_Rept_Positions_Short_ALL"),
        "managed_money_weekly_net_change": ("Change_in_M_Money_Long_All", "Change_in_M_Money_Short_All"),
        "commercial_weekly_net_change": ("Change_in_Comm_Long_All", "Change_in_Comm_Short_All"),
        "noncommercial_weekly_net_change": ("Change_in_NonComm_Long_All", "Change_in_NonComm_Short_All"),
    }

    derived_features = {}
    for new_col, (long_col, short_col) in formulas.items():
        if long_col in cot.columns and short_col in cot.columns:
            long_values = pd.to_numeric(cot[long_col], errors="coerce")
            short_values = pd.to_numeric(cot[short_col], errors="coerce")
            derived_features[new_col] = long_values - short_values

    if "Change_in_Open_Interest_All" in cot.columns and "Open_Interest_All" in cot.columns:
        change_oi = pd.to_numeric(cot["Change_in_Open_Interest_All"], errors="coerce")
        open_interest = pd.to_numeric(cot["Open_Interest_All"], errors="coerce")
        derived_features["open_interest_change_pct"] = change_oi / open_interest

    if derived_features:
        cot = pd.concat([cot, pd.DataFrame(derived_features, index=cot.index)], axis=1).copy()

    return cot


yahoo_features = add_yahoo_derived_features(yahoo_prices)
cot_features_enriched = add_cot_derived_features(cot_features)

print(f"Yahoo feature columns: {len(yahoo_features.columns):,}")
print(f"COT feature columns: {len(cot_features_enriched.columns):,}")


In [ ]:
def full_outer_join_yahoo_and_cot(yahoo, cot):
    yahoo_for_join = yahoo.copy()
    cot_for_join = cot.copy()

    merged = yahoo_for_join.merge(
        cot_for_join,
        how="outer",
        left_on="Date",
        right_on="cot_report_date",
        indicator=True,
        suffixes=("_yahoo", "_cot"),
    )

    merged["Date"] = merged["Date"].combine_first(merged["cot_report_date"])
    merged["date_match_status"] = merged["_merge"].map(
        {
            "both": "both",
            "left_only": "yahoo_only",
            "right_only": "cot_only",
        }
    )

    sort_columns = ["Date", "source_dataset", "source_archive"]
    sort_columns = [col for col in sort_columns if col in merged.columns]
    merged = merged.sort_values(sort_columns, na_position="last").reset_index(drop=True)
    merged = merged.drop(columns=["_merge"])

    first_cols = ["Date", "date_match_status", "cot_report_date"]
    first_cols = [col for col in first_cols if col in merged.columns]
    remaining_cols = [col for col in merged.columns if col not in first_cols]
    merged = merged[first_cols + remaining_cols]

    return merged


central_data = full_outer_join_yahoo_and_cot(yahoo_features, cot_features_enriched)

central_data_file = CENTRAL_DATA_DIR / "yahoo_cot_full_outer_by_date.csv"
central_data.to_csv(central_data_file, index=False)

join_summary = (
    central_data["date_match_status"]
    .value_counts(dropna=False)
    .rename_axis("date_match_status")
    .reset_index(name="rows")
)
join_summary_file = CENTRAL_DATA_DIR / "yahoo_cot_date_join_summary.csv"
join_summary.to_csv(join_summary_file, index=False)

unmatched_dates = central_data.loc[
    central_data["date_match_status"].isin(["yahoo_only", "cot_only"]),
    ["Date", "date_match_status", "source_dataset", "source_archive", "Market_and_Exchange_Names"],
].copy()
unmatched_dates_file = CENTRAL_DATA_DIR / "yahoo_cot_unmatched_dates.csv"
unmatched_dates.to_csv(unmatched_dates_file, index=False)

print(f"Wrote central joined data: {central_data_file}")
print(f"Wrote join summary: {join_summary_file}")
print(f"Wrote unmatched dates: {unmatched_dates_file}")
print(join_summary.to_string(index=False))
print(f"Final rows: {len(central_data):,}")
print(f"Final columns: {len(central_data.columns):,}")


In [5]:
central_data_file = CENTRAL_DATA_DIR / "yahoo_cot_full_outer_by_date.csv"
central_data = pd.read_csv(central_data_file)
central_data.head(10)



/var/folders/pr/sfsw3lhs1bd3dyyn157lt0g80000gn/T/ipykernel_68288/4207424260.py:2: DtypeWarning: Columns (0: future_direction_5d) have mixed types. Specify dtype option on import or set low_memory=False.
  central_data = pd.read_csv(central_data_file)


,Date,date_match_status,cot_report_date,Close,High,Low,Open,Volume,return_1d,return_5d,...,producer_merchant_net,commercial_net,noncommercial_net,nonreportable_net,swap_dealer_net,other_reportable_net,managed_money_weekly_net_change,commercial_weekly_net_change,noncommercial_weekly_net_change,open_interest_change_pct
0,2000-01-03,yahoo_only,NaN,116.500000,124.000000,116.099998,124.00,6640.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,both,2000-01-04,116.250000,120.500000,115.750000,116.50,5492.0,-0.002146,NaN,...,NaN,-13595.0,10529.0,3066.0,NaN,NaN,NaN,721.0,-484.0,0.023446
2,2000-01-05,yahoo_only,NaN,118.599998,121.000000,115.000000,115.00,6165.0,0.020215,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,yahoo_only,NaN,116.849998,121.400002,116.500000,119.00,5094.0,-0.014755,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,yahoo_only,NaN,114.150002,117.750000,113.800003,117.75,6855.0,-0.023107,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2000-01-10,yahoo_only,NaN,117.550003,126.000000,116.699997,126.00,7499.0,0.029785,0.009013,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2000-01-11,both,2000-01-11,117.800003,118.250000,115.500000,115.50,3976.0,0.002127,0.013333,...,NaN,-12874.0,9672.0,3202.0,NaN,NaN,NaN,721.0,-857.0,0.016641
7,2000-01-12,yahoo_only,NaN,118.949997,120.500000,116.900002,118.00,5184.0,0.009762,0.002951,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2000-01-13,yahoo_only,NaN,118.550003,120.000000,117.500000,120.00,3717.0,-0.003363,0.014549,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2000-01-14,yahoo_only,NaN,112.550003,120.250000,112.250000,118.00,10115.0,-0.050612,-0.014017,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
